Loading Data

In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

### Data Loading
pdf_path = Path('data/jp_morgan_supplier_llm.pdf')
if not pdf_path.exists():
    pdf_path = (Path.cwd().resolve().parent / 'data' / 'jp_morgan_supplier_llm.pdf').resolve()

if not pdf_path.exists():
    raise FileNotFoundError(f"Document not found: {pdf_path}")

loader = PyPDFLoader(str(pdf_path))
pdf_document = loader.load()
pdf_document

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '2025-10-01T00:50:47+00:00', 'author': 'Wanying Ding; Savinay Narendra; Xiran Shi; Adwait Ratnaparkhi; Chengrui Yang; Nikoo Sabzevar; Ziyan Yin', 'doi': 'https://doi.org/10.48550/arXiv.2509.25803', 'keywords': 'Financial Transaction Understanding, Merchant Retrieval, Transformer Model, Large Language Model, Natural Language Understanding', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'moddate': '2025-10-01T00:50:47+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Better with Less: Small Proprietary Models Surpass Large Language Models in Financial Transaction Understanding', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2509.25803v1', 'source': 'data\\jp_morgan_supplier_llm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Better with Less: Small Proprietary Mo

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
text_chunks = text_splitter.split_documents(pdf_document)
text_chunks

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '2025-10-01T00:50:47+00:00', 'author': 'Wanying Ding; Savinay Narendra; Xiran Shi; Adwait Ratnaparkhi; Chengrui Yang; Nikoo Sabzevar; Ziyan Yin', 'doi': 'https://doi.org/10.48550/arXiv.2509.25803', 'keywords': 'Financial Transaction Understanding, Merchant Retrieval, Transformer Model, Large Language Model, Natural Language Understanding', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'moddate': '2025-10-01T00:50:47+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Better with Less: Small Proprietary Models Surpass Large Language Models in Financial Transaction Understanding', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2509.25803v1', 'source': 'data\\jp_morgan_supplier_llm.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Better with Less: Small Proprietary Mo

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

Vector Embedding Techniques and Vector Stores

In [4]:
#### for creating vectors
from langchain_community.embeddings import OpenAIEmbeddings 

### for storing vectors in a vector database, we can use chromadb, pinecone, 
# weaviate, etc. Here we will use chromadb for simplicity
from langchain_community.vectorstores import Chroma

db_chroma = Chroma.from_documents(text_chunks, OpenAIEmbeddings())


C:\Users\prash\AppData\Local\Temp\ipykernel_39344\2027295583.py:8: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  db_chroma = Chroma.from_documents(text_chunks, OpenAIEmbeddings())


In [5]:
### Vector Db

query = "Who is the author of the document?"
result = db_chroma.similarity_search(query)
result

[Document(metadata={'creationdate': '2025-10-01T00:50:47+00:00', 'doi': 'https://doi.org/10.48550/arXiv.2509.25803', 'producer': 'pikepdf 8.15.1', 'source': 'data\\jp_morgan_supplier_llm.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'page': 7, 'keywords': 'Financial Transaction Understanding, Merchant Retrieval, Transformer Model, Large Language Model, Natural Language Understanding', 'title': 'Better with Less: Small Proprietary Models Surpass Large Language Models in Financial Transaction Understanding', 'total_pages': 9, 'author': 'Wanying Ding; Savinay Narendra; Xiran Shi; Adwait Ratnaparkhi; Chengrui Yang; Nikoo Sabzevar; Ziyan Yin', 'license': 'http://creativecommons.org/licenses/by-nc-nd/4.0/', 'moddate': '2025-10-01T00:50:47+00:00', 'creator': 'arXiv GenPDF (tex2pdf:)', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2509.25803v1', 'page_label': '8'}, page_content='Research25, 70 (2024), 1–53.\n[4] Tim 

In [6]:
#### FAISS Vector Store
from langchain_community.vectorstores import FAISS 
db_faiss = FAISS.from_documents(text_chunks, OpenAIEmbeddings())

In [7]:
query = "Who is the author of the document?"
result = db_faiss.similarity_search(query)
result[0].page_content

'Research25, 70 (2024), 1–53.\n[4] Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. 2024.\nQlora: Efficient finetuning of quantized llms.Advances in Neural Information\nProcessing Systems36 (2024).\n2https://slopepay.com/company\n3https://medium.com/slope-stories/slopegpt-the-first-payments-risk-model-\npowered-by-gpt-4-cd444ab5242d\n4https://medium.com/slope-stories/slope-transformer-the-first-llm-trained-to-\nunderstand-the-language-of-banks-88adbb6c8da9'

In [8]:
from langchain_community.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

C:\Users\prash\AppData\Local\Temp\ipykernel_39344\1591636548.py:2: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)


In [9]:
## Design Chat Prompt Template
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Use the following context to answer the question.\n"
    "Context:\n{context}\n"
    "Question: {question}\n"
    "Answer:"
    """
 )

In [10]:
# ## Chain Induction
# ## Create Stuff Documents Chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
# doc_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

# The import from langchain.chains.combine_documents import create_stuff_documents_chain 
# is incorrect because, in langchain version 1.2.13, the module combine_documents 
# and the function create_stuff_documents_chain no longer exist. 
# The langchain library has changed its API, and this function was removed or refactored 
# in recent versions.


In [17]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import PromptTemplate

# Example prompt and LLM setup
prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Use the following context to answer the question.\n"
    "Context:\n{context}\n"
    "Question: {question}\n"
    "Answer:"
    """
 )
# llm = ... (your LLM setup)

def stuff_chain(inputs):
    docs = inputs["docs"]
    question = inputs["question"]
    # Handle both Document objects and plain strings
    if isinstance(docs, list) and len(docs) > 0:
        if isinstance(docs[0], str):
            context = "\n".join(docs)
        else:
            context = "\n".join([doc.page_content for doc in docs])
    else:
        context = ""
    print("Context passed to LLM:\n", context)  # Debug print
    return {"context": context, "question": question}

chain = RunnableLambda(stuff_chain) | prompt | llm

In [18]:
retriever_c = db_chroma.as_retriever()
retriever_c

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000021582CE1D90>, search_kwargs={})

In [19]:
retriever_f = db_faiss.as_retriever()
retriever_f

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002158581A360>, search_kwargs={})

In [20]:
from langchain.chains import CreateRetrievalChain
retrieval_chain_c = CreateRetrievalChain.from_retriever(retriever_c, chain)

ModuleNotFoundError: No module named 'langchain.chains'

In [21]:
from langchain_core.runnables import RunnableLambda

# Assume retriever_c is your retriever and chain is your stuff chain (Runnable)
# Use retriever_c as a Runnable directly
# This will pass the question to the retriever and get documents, then stuff them into the chain
def retrieve_context(inputs):
    docs = retriever_c.invoke(inputs["question"])
    # If docs are strings, just join them
    if isinstance(docs[0], str):
        context = "\n".join(docs)
    else:
        context = "\n".join([doc.page_content for doc in docs])
    return {"context": context, "question": inputs["question"]}

retrieval_chain_c = RunnableLambda(retrieve_context) | chain

In [22]:
docs = retriever_c.invoke("Who is the author of the document?")
response = chain.invoke({"docs": docs, "question": "Who is the author of the document?"})
print(response)

Context passed to LLM:
 Research25, 70 (2024), 1–53.
[4] Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. 2024.
Qlora: Efficient finetuning of quantized llms.Advances in Neural Information
Processing Systems36 (2024).
2https://slopepay.com/company
3https://medium.com/slope-stories/slopegpt-the-first-payments-risk-model-
powered-by-gpt-4-cd444ab5242d
4https://medium.com/slope-stories/slope-transformer-the-first-llm-trained-to-
understand-the-language-of-banks-88adbb6c8da9
Better with Less: Small Proprietary Models Surpass Large
Language Models in Financial Transaction Understanding
Wanying Ding
JPMorgan Chase & Co.
Palo Alto, California, USA
wanying.ding@jpmchase.com
Savinay Narendra
JPMorgan Chase & Co.
Palo Alto, California, USA
savinay.narendra@jpmchase.com
Xiran Shi
JPMorgan Chase & Co.
Palo Alto, California, USA
xiran.shi@jpmchase.com
Adwait Ratnaparkhi
JPMorgan Chase & Co.
Palo Alto, California, USA
adwait.ratnaparkhi@jpmchase.com
Chengrui Yang
JPMorgan Chase & 

In [23]:
# Retrieve documents and run the chain for a question
question = "Who is the author of the document?"
docs = retriever_c.invoke(question)
response = chain.invoke({"docs": docs, "question": question})
print(response)

Context passed to LLM:
 Research25, 70 (2024), 1–53.
[4] Tim Dettmers, Artidoro Pagnoni, Ari Holtzman, and Luke Zettlemoyer. 2024.
Qlora: Efficient finetuning of quantized llms.Advances in Neural Information
Processing Systems36 (2024).
2https://slopepay.com/company
3https://medium.com/slope-stories/slopegpt-the-first-payments-risk-model-
powered-by-gpt-4-cd444ab5242d
4https://medium.com/slope-stories/slope-transformer-the-first-llm-trained-to-
understand-the-language-of-banks-88adbb6c8da9
Better with Less: Small Proprietary Models Surpass Large
Language Models in Financial Transaction Understanding
Wanying Ding
JPMorgan Chase & Co.
Palo Alto, California, USA
wanying.ding@jpmchase.com
Savinay Narendra
JPMorgan Chase & Co.
Palo Alto, California, USA
savinay.narendra@jpmchase.com
Xiran Shi
JPMorgan Chase & Co.
Palo Alto, California, USA
xiran.shi@jpmchase.com
Adwait Ratnaparkhi
JPMorgan Chase & Co.
Palo Alto, California, USA
adwait.ratnaparkhi@jpmchase.com
Chengrui Yang
JPMorgan Chase & 